# CoraML Face Dataset Demo

Inspect the facial-walk dataset and chunked dataloader on the CoraML graph.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import torch

from facialgen.data import (
    CyclicFaceChunkDataset,
    FacialWalkVertexDataset,
    load_coraml_sparse,
    make_face_chunk_dataloader,
)
from facialgen.curvature import largest_connected_component, resistance_curvature

A_full, X, y = load_coraml_sparse(data_dir="data")
A_lcc, nodes_lcc = largest_connected_component(A_full)
n_lcc = A_lcc.shape[0]

curvature = resistance_curvature(
    A_lcc,
    use_lcc=False,
    solver="lstsq",
    rhs_scale=float(n_lcc),
)

face_ds = FacialWalkVertexDataset(
    A_lcc,
    curvature,
    num_sign_configs=3,
    sign_seed=2026,
)

chunk_ds = CyclicFaceChunkDataset(
    face_ds,
    context_size=32,
    epoch_seed=99,
)

print(f"CoraML full graph: {A_full.shape}, nnz={A_full.nnz}")
print(f"CoraML LCC: {A_lcc.shape}, nnz={A_lcc.nnz}")
print(f"num full face sequences: {len(face_ds)}")
print(f"num chunk samples @ T=32: {len(chunk_ds)}")
print(f"vocab_size={face_ds.vocab_size}, BOS={face_ds.bos_token_id}, EOS={face_ds.eos_token_id}, PAD={chunk_ds.pad_token_id}")

full0 = face_ds[0]
print("\nfull face sample 0")
print({
    "sequence_length": full0["sequence_length"],
    "sign_config_index": full0["sign_config_index"],
    "face_index_within_config": full0["face_index_within_config"],
})
print(full0["tokens"][:40])

chunk_ds.set_epoch(0)
chunk0_e0 = chunk_ds[0]
chunk_ds.set_epoch(1)
chunk0_e1 = chunk_ds[0]

print("\nchunk sample 0 at epoch 0")
print({k: v for k, v in chunk0_e0.items() if k != "tokens"})
print(chunk0_e0["tokens"])

print("\nchunk sample 0 at epoch 1")
print({k: v for k, v in chunk0_e1.items() if k != "tokens"})
print(chunk0_e1["tokens"])

chunk_ds.set_epoch(1)
loader = make_face_chunk_dataloader(chunk_ds, batch_size=4, shuffle=False)
batch = next(iter(loader))

print("\ncollated batch keys")
print(sorted(batch.keys()))
print("\ninput_ids shape:", tuple(batch["input_ids"].shape))
print("attention_mask shape:", tuple(batch["attention_mask"].shape))
print("labels shape:", tuple(batch["labels"].shape))
print("lengths:", batch["lengths"])
print("\ninput_ids")
print(batch["input_ids"])
print("\nattention_mask")
print(batch["attention_mask"].to(torch.int64))
print("\nlabels")
print(batch["labels"])


CoraML full graph: (2995, 2995), nnz=16316
CoraML LCC: (2810, 2810), nnz=15962
num full face sequences: 657
num chunk samples @ T=32: 2052
vocab_size=2812, BOS=2810, EOS=2811, PAD=2812

full face sample 0
{'sequence_length': 14556, 'sign_config_index': 0, 'face_index_within_config': 0}
tensor([2810,    0, 1581, 2241, 1581, 1668, 1672, 1582, 1370, 1479, 1676, 1675,
        1258, 2399,  365, 1275, 1352,  365, 1262, 1275, 1316, 1259, 1484, 2614,
        1272, 1285, 1248, 1284, 1300, 1285, 1302, 1248, 1300, 1302, 1274, 1310,
        1320, 2261, 1550, 2261])

chunk sample 0 at epoch 0
{'face_index': 0, 'chunk_index': 0, 'num_chunks_for_face': 455, 'sign_config_index': 0, 'face_index_within_config': 0}
tensor([2810, 2643, 2654,  892, 2643, 2644, 2649, 2647, 2643, 2656, 2643,  892,
        2661,  892, 2101,  892, 2646,  700,  892, 2654,  700, 2646,  892,  700,
        2654, 2663, 2660, 2663, 2654, 2643, 2647, 2638])

chunk sample 0 at epoch 1
{'face_index': 0, 'chunk_index': 0, 'num_chunks_fo